# Merge TCIA metadata with mask statistics

In [62]:
import pandas as pd
import re

In [ ]:
# load data
metadata = pd.read_csv("../../../data/raw/metadata/merged_ku_tcia.csv")
final_dataset = pd.read_csv("../../../data/processed/final_dataset/final_dataset_statistics.csv")

In [84]:
# extract old filenames
def extract_sub_id(filename):
    match_1 = re.match(r"colon_(0*\d+)-", filename)
    match_2 = re.match(r"^colon_\d+-(prone|supine).*\.mha$", filename)

    if match_1 and match_2:
        # sub001_pos-prone_scan-1
        return f"sub{int(match_1.group(1)):03d}_pos-{match_2.group(1)}_scan-1"
    return None

final_dataset["name"] = final_dataset["filename"].apply(extract_sub_id)

# merge
final_dataset_merged = pd.merge(
    final_dataset,
    metadata[["Patient ID", "Patient Sex", "Patient Age", "position", "name", "Series Instance UID", "Manufacturer", "Manufacturer Model Name", "Software Versions"]],
    on="name",
    how="left"
)

# rename column header
final_dataset_merged = final_dataset_merged.rename(columns={"Patient ID": "patient_id", "Patient Sex": "patient_sex", "Patient Age": "patient_age", "position": "patient_position", "Series Instance UID": "instance_uid", "Manufacturer": "manufacturer", "Manufacturer Model Name": "manufacturer_model_name", "Software Versions": "software_versions"})
final_dataset_merged = final_dataset_merged.drop(columns="name")

In [ ]:
final_dataset_merged.to_csv("../../../data/processed/final_dataset/final_dataset_statistics_and_metadata.csv")

In [ ]:
# check for nan values
nan_val = final_dataset_merged[final_dataset_merged["patient_position"].isna()]
nan_val

,filename,component_id,voxel_count,skeleton_size,patient_id,patient_sex,patient_age,patient_position,instance_uid,manufacturer,manufacturer_model_name,software_versions
273,colon_0136-supine.mha,1,1210267.0,4271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
340,colon_0169-prone.mha,1,458411.0,822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,colon_0169-prone.mha,2,767591.0,1331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
342,colon_0169-prone.mha,3,913875.0,1584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
343,colon_0169-prone.mha,4,3012.0,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
512,colon_0231-prone.mha,1,1784155.0,2178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
686,colon_0302-supine.mha,1,106302.0,365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
687,colon_0302-supine.mha,2,1340820.0,2153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
688,colon_0302-supine.mha,3,10594.0,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
794,colon_0348-prone.mha,1,1134595.0,4130,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# TODO: check for overlap
# {"InstanceUID":"1.3.6.1.4.1.9328.50.4.263216","Sex":"F","Position":"supine","subject_id":"sub616","number":335,"nnunet_image_file":"colon_335_0000.mha","nnunet_label_file":"colon_335.mha"}

# unique identifier? Series Instance UID

# ls ./data_temp/converted/sub136
# sub136_pos-prone_scan-1_conv-sitk.mha.gz  sub136_pos-supine_scan-1_conv-sitk.mha.gz  sub136_pos-supine_scan-2_conv-sitk.mha.gz


# columns = ["Patient Sex", "Patient Age", "Study Instance UID", "Manufacturer", "Manufacturer Model Name", "Software Versions"]

# metadata_tcia = pd.read_excel("../../../data/raw/metadata/metadata_tcia.xlsx")
# metadata_tcia = metadata_tcia[["Patient ID", "Patient Sex", "Patient Age", "Study Instance UID", "Manufacturer", "Manufacturer Model Name", "Software Versions"]]

# for column in columns:
#     grouped_metadata = metadata_tcia.groupby(by="Patient ID")[column].unique()
#     for v in grouped_metadata.values:
#         if len(v) > 1:
#             print(v)